# EnaOS Headless Capture & Demo Environment

This notebook compiles and runs **EnaOS** in a headless Linux environment (like Google Colab). It launches a headless Wayland compositor (Sway), starts the `enad` system daemon, boots the `ena-bar` GTK4 UI bar, and captures screenshots in different states using `grim`.

### System Architecture:
1. **Headless Wayland Server:** `sway` configured with the `headless` backend.
2. **enad (Rust Daemon):** Integrates with the headless compositor, monitors system state, and routes IPC messages.
3. **ena-bar (GTK4 Bar):** Runs natively inside the compositor, rendering the UI overlay.
4. **AI Runtime (Python):** Standalone FastAPI server routing queries to Ollama/cloud.
5. **Screen Capturer (`grim`):** Takes screenshots of the virtual output and displays them inline.

## Step 1: Install System Dependencies

We need to install Sway, Weston (for `weston-terminal`), compilation toolchains, and GTK4 / Layer Shell libraries.

In [ ]:
# Install dependencies for Wayland, GTK4, Layer Shell, and compilation
!sudo apt-get update -y
!sudo apt-get install -y sway weston grim build-essential pkg-config git \
    libgtk-4-dev libadwaita-1-dev libgtk4-layer-shell-dev python3-pip python3-venv xdg-desktop-portal

## Step 2: Install Rust Toolchain

EnaOS's core daemon and GUI bar are written in Rust. Let's install the compiler.

In [ ]:
# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"
!rustc --version

## Step 3: Clone EnaOS Repository (if not already local)

This cell handles cloning EnaOS if you're running this notebook directly in Google Colab.

In [ ]:
import os
if not os.path.exists("EnaOS") and not os.path.exists("runtimes/enad"):
    !git clone https://github.com/anshull-saxena/EnaOS.git
    os.chdir("EnaOS")
elif os.path.exists("EnaOS"):
    os.chdir("EnaOS")
print(f"Current working directory: {os.getcwd()}")

## Step 4: Build EnaOS Components

Compile the daemon (`enad`) and the native bar UI (`ena-bar`).

In [ ]:
# Build enad daemon and ena-bar GTK4 shell
!cargo build --manifest-path runtimes/enad/Cargo.toml --release
!cargo build --manifest-path shell/ena-bar/Cargo.toml --release

## Step 5: Setup Python AI Runtime Virtual Environment

Set up dependencies for the FastAPI AI server.

In [ ]:
# Set up AI runtime venv
!python3 -m venv runtimes/ai-runtime/.venv
!runtimes/ai-runtime/.venv/bin/pip install -r runtimes/ai-runtime/requirements.txt

## Step 6: Start Headless Sway Wayland Compositor

We create a minimal configuration file specifying a headless output resolution, and launch Sway in the background using the wlroots headless backend.

In [ ]:
import subprocess
import time
import os

# Write minimal Sway headless configuration
sway_config = """
# Minimal Sway Headless Config
output HEADLESS-1 resolution 1280x720
workspace_layout tabbed
default_border none
default_floating_border none
client.focused #D4AF37 #171717 #ffffff #D4AF37 #D4AF37
"""
with open("/tmp/sway.config", "w") as f:
    f.write(sway_config)

# Setup Wayland runtime dir
os.environ["XDG_RUNTIME_DIR"] = "/tmp/xdg"
os.makedirs("/tmp/xdg", exist_ok=True)
os.chmod("/tmp/xdg", 0o700)
os.environ["WLR_BACKENDS"] = "headless"
os.environ["WLR_LIBINPUT_NO_DEVICES"] = "1"

print("Starting headless Sway compositor...")
sway_proc = subprocess.Popen(
    ["sway", "-c", "/tmp/sway.config"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Allow Sway to boot and bind its sockets
time.sleep(3)

# Discover Wayland sockets
sway_socket = None
for root, dirs, files in os.walk("/tmp/xdg"):
    for f in files:
        if f.startswith("sway-ipc"):
            sway_socket = os.path.join(root, f)
            break

if not sway_socket:
    uid = os.getuid()
    user_runtime = f"/run/user/{uid}"
    if os.path.exists(user_runtime):
        for root, dirs, files in os.walk(user_runtime):
            for f in files:
                if f.startswith("sway-ipc"):
                    sway_socket = os.path.join(root, f)
                    break

if sway_socket:
    print(f"Found Sway IPC Socket: {sway_socket}")
    os.environ["SWAYSOCK"] = sway_socket
    # Cache env variables so subsequent subprocesses can read them
    with open("/tmp/sway.env", "w") as f:
        f.write(f"export SWAYSOCK={sway_socket}\n")
        f.write(f"export WAYLAND_DISPLAY=wayland-1\n")
        f.write(f"export XDG_RUNTIME_DIR=/tmp/xdg\n")
        f.write(f"export WLR_BACKENDS=headless\n")
        f.write(f"export WLR_LIBINPUT_NO_DEVICES=1\n")
else:
    print("WARNING: Sway IPC socket not found! Focus tracking and window management may fail.")

## Step 7: Launch EnaOS Core Daemon

Start `enad` which listens on `/tmp/enad.sock` for client connections.

In [ ]:
# Start enad
if os.path.exists("/tmp/enad.sock"):
    try:
        os.remove("/tmp/enad.sock")
    except OSError:
        pass

print("Starting enad system daemon...")
enad_proc = subprocess.Popen(
    ["./runtimes/enad/target/release/enad", "--socket", "/tmp/enad.sock"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(2)
print("enad socket running.")

## Step 8: Start AI Runtime

Boot the Python AI runtime to handle prompt context parsing and REST endpoints.

In [ ]:
# Start AI runtime
print("Starting AI Runtime...")
ai_proc = subprocess.Popen(
    ["./runtimes/ai-runtime/.venv/bin/python", "-m", "src.main"],
    cwd="./runtimes/ai-runtime",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(2)
print("AI Runtime running on http://localhost:8900")

## Step 9: Launch Ena Bar UI inside Sway

Send the execution request to Sway to spawn the GTK4 layer-shell bar.

In [ ]:
# Run ena-bar in the active Wayland session
env_cmd = ""
if os.path.exists("/tmp/sway.env"):
    env_cmd = "source /tmp/sway.env && "

print("Launching ena-bar...")
!{env_cmd}swaymsg exec "./shell/ena-bar/target/release/ena-bar --socket-path /tmp/enad.sock"
time.sleep(2)
print("ena-bar is running.")

## Step 10: Setup Capture and Socket Helpers

Define python helpers to communicate directly with the daemon's Unix socket and capture the virtual output frame.

In [ ]:
import socket
import json
import uuid
from IPython.display import Image, display

def send_command(command_type, body=None):
    """Utility function to send IPC command messages to enad."""
    s = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    try:
        s.connect("/tmp/enad.sock")
        msg = {
            "id": str(uuid.uuid4()),
            "kind": {
                "type": "Command",
                "body": {
                    command_type: body
                } if body is not None else command_type
            }
        }
        s.sendall((json.dumps(msg) + "\n").encode())
        response = s.recv(4096).decode()
        return json.loads(response)
    except Exception as e:
        return {"error": str(e)}
    finally:
        s.close()

def capture_screenshot(filename):
    """Take a framebuffer capture using grim and display it in-line."""
    env_cmd = ""
    if os.path.exists("/tmp/sway.env"):
        env_cmd = "source /tmp/sway.env && "
    
    # Capture frame
    !{env_cmd}grim {filename}
    
    print(f"Saved screenshot: {filename}")
    display(Image(filename))

## Step 11: Capture Screenshots

### State 1: Fresh Onboarding Overlay Screen
Because onboarding hasn't been completed yet, the bar will automatically open the first-run welcome window.

In [ ]:
# Capture welcome overlay
capture_screenshot("ena-bar-welcome.png")

### State 2: Onboarding Complete & Collapsed Bar
Dismiss the overlay via IPC command and capture the minimal bar state showing just the status dot.

In [ ]:
# Complete onboarding
print("Dismissing onboarding...")
print(send_command("CompleteOnboarding"))
time.sleep(2)

# Capture collapsed state
capture_screenshot("ena-bar-collapsed.png")

### State 3: Active Window Focus Tracker
Launch a GUI app (`weston-terminal`) inside Sway. The daemon will capture the focus shift and update the context label.

In [ ]:
# Start terminal to trigger focus tracking update
env_cmd = ""
if os.path.exists("/tmp/sway.env"):
    env_cmd = "source /tmp/sway.env && "

print("Launching weston-terminal...")
!{env_cmd}swaymsg exec "weston-terminal"
time.sleep(2)

# Capture screen with focused app
capture_screenshot("ena-bar-focused.png")

## Step 12: Clean Up Processes

Stop all backgrounds processes cleanly.

In [ ]:
# Stop processes
print("Stopping Sway, enad, and AI runtime...")
try:
    sway_proc.terminate()
    enad_proc.terminate()
    ai_proc.terminate()
    print("Cleanup complete.")
except Exception as e:
    print(f"Cleanup error: {e}")